# Gene prioritization
---
1. Knowledge graph generation
2. Edges train-validation-test split
3. Node embeddings generation on training edges with node2vec
4. Edge embeddings generation by taking the Hadamard product of node embeddings
5. Cross-validation by link prediction task on validation and test edges with loggistic regression
6. Node embeddings generation with tuned hyper-parameters on all edges with node2vec

In [7]:
import os
import pandas as pd
import pickle
from CADA.triples_gene_hpo import triples_gene_hpo
import networkx as nx
import numpy as np
from CADA.gae.preprocessing import mask_test_edges
from gensim.models import Word2Vec
from CADA.paths import DATA_DIRECTORY,MODEL_DIRECTORY

In [8]:
# Collect only gene-hpo term edges
gene_hpo_triples = triples_gene_hpo()
gene_hpo_edges = []
for triple in gene_hpo_triples:
    gene_hpo_edges.append((triple[0], triple[2]))
g_gene_hpo_edges = nx.Graph()
g_gene_hpo_edges.add_edges_from(gene_hpo_edges)
adj_sparse_gene_hpo_edges = nx.to_scipy_sparse_matrix(g_gene_hpo_edges)

# record gene nodes
nodes = list(g_gene_hpo_edges.nodes())
nodes_gene = []
for node in nodes:
    if node.startswith("Entrez:"):
        nodes_gene.append(node)

In [9]:
# get embeddings and create node embeddings matrix(rows = nodes, columns = embedding features)
model = Word2Vec.load('results/node2vec.model')
emb_mappings = model.wv
emb_mappings

In [10]:
#count the number of training patients with a specific mutation for each mutation gene
train = 'results/patient_training.tsv'
if os.path.exists(train):
    train_pd = pd.read_csv(train, header=None, sep='\t', names=['patient_id', 'omim_id', 'gene_id', 'features', 'submitter',
                                    'from_file', 'no_features'])
    gene_counts = train_pd['gene_id'].value_counts().to_dict() 
else:
    gene_counts = {}

In [11]:
with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'gene_id_name.dict'), 'rb') as handle:
    gene_id_name = pickle.load(handle)

with open(os.path.join(DATA_DIRECTORY, 'processed', 'ids', 'hpo_id_name.dict'), 'rb') as handle:
    hpo_id_name = pickle.load(handle)

In [12]:
## Prioritizing genes for each test patient based on their hpo terms
with open('patient_testing.tsv', 'r') as t_file:
    content = t_file.read().splitlines()
    content = [x.split('\t') for x in content]
    evaluation_save = list()
    evaluation_vis = list()
    for line in content:
        prioritized_genelist = []
        patient_id = line[0]
        gene_id = line[2]
        # Get the number training patients with same mutation gene
        if gene_id in gene_counts:
            no_patients = gene_counts[gene_id]
        else:
            no_patients = 0
        
        # prioritize genes for each patients by computing the similarity between genes and hpo terms by dot product
        features = line[3].split(',')
        table = {}
        for gene in nodes_gene:
            sc_scores = []
            gene_emb = model.wv[gene]
            for feature in features:
                feature_emb = model.wv[feature]
                sc_score = np.dot(feature_emb, gene_emb)
                sc_scores.append(sc_score)
            table[gene] = sum(sc_scores)
        prioritized_genes = pd.DataFrame.from_dict(table, orient='index', columns=['score']).sort_values(by='score', ascending=False).index.values.tolist()
        # get the rank
        if gene_id in prioritized_genes:
            rank = prioritized_genes.index(gene_id) + 1
        else:
            rank = 'NA'
        evaluation_save.append([patient_id, gene_id, no_patients,','.join(features), prioritized_genes[:30], rank])
        evaluation_vis.append([patient_id, gene_id_name[gene_id], no_patients,
                                   ','.join([hpo_id_name[feature] for feature in features]), ','.join([gene_id_name[gene_id] for gene_id in prioritized_genes[:30]]), rank])
    # save the result
    saveframe = pd.DataFrame(evaluation_save, columns = ['patient_id', 'gene_id','no_patients', 'features', 'result', 'rank'])
    saveframe.to_csv('evaluation.tsv', sep='\t', index=None)
    visframe = pd.DataFrame(evaluation_vis, columns=['patient_id', 'gene', 'no_patients', 'features', 'result', 'rank'])
    visframe.to_excel('evaluation.xlsx', index=None) 
                
    

KeyboardInterrupt: 

In [ ]:
# count top-n accuracy 
ranks = saveframe['rank'].tolist()
count1 = count5 = count10 = count50 = count100 = count1000 = 0
counts = len(ranks)
for rank in ranks:
    if isinstance(rank, int):
        if rank == 1:
            count1 += 1
        if rank <= 5:
            count5 += 1
        if rank <= 10:
            count10 += 1
        if rank <= 50:
            count50 += 1
        if rank <= 100:
            count100 += 1
        if rank <= 1000:
            count1000 += 1
top1 = round(count1/counts, 3)
top5 = round(count5/counts, 3)
top10 = round(count10/counts, 3)
top50 = round(count50/counts, 3)
top100 = round(count100/counts, 3)
top1000 = round(count1000/counts, 3)

print(f'top1: {top1}')
print(f'top5: {top5}')
print(f'top10: {top10}')
print(f'top50: {top50}')
print(f'top100: {top100}')
print(f'top1000: {top1000}')

In [ ]:
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt


def autolabel(rects):
    """Attach a text label above each bar in *rects*, displaying its height."""
    for rect in rects:
        height = rect.get_height()
        ax.annotate('{}'.format(height),
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')


def mkdir_p(path):
    import os
    if not os.path.isdir(path):
        os.makedirs(path)


sns.set_style('white')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['font.size'] = 11
fig, ax = plt.subplots(1)

width=0.3
distance = 2
labels = ['Top1', 'Top5', 'Top10', 'Top50', 'Top100', 'Top1000']
values = [round(i*100,1) for i in [top1 , top5, top10, top50, top100, top1000]]
x =np.arange(len(labels))
rects = ax.bar(x, values, width,color='lightblue', edgecolor=(96/255.0, 133/255.0, 131/255.0))
autolabel(rects)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Ranked Gene Lists", fontsize=15)
ax.set_ylabel("% of Cases with Causal Gene in List", fontsize=14.5)

sns.despine()

mkdir_p("figures")
plt.savefig('figures/topn.png', bbox_inches='tight')


In [6]:
# count median rank for patients of each gene
gene_rank = saveframe.groupby('gene_id')[['rank']].median().sort_values(by='rank')
gene_rank.columns = ['median_rank']
gene_rank

NameError: name 'saveframe' is not defined

In [ ]:
gene_analysis = pd.read_csv('../descriptive_statistics/genes_hpo_disease_patient.csv', sep = '\t', index_col=0)

In [ ]:
gene_table = pd.merge(gene_analysis, gene_rank, left_index=True, right_index=True, how='outer')

In [ ]:
gene_table

In [ ]:
gna_row = gene_table[gene_table.isna().any(axis=1)]
na_row

In [ ]:
import seaborn as sns

In [ ]:
sns.lmplot(x='num_hpo_annotation', y='median_rank', data=gene_table, fit_reg=True)